# Staging — OECD IP charges

Parses the raw SDMX-JSON response into a clean, flat CSV.

| | |
|---|---|
| **Reads** | `data/raw/oecd/raw_oecd__ip_charges.json` |
| **Writes** | `data/staging/oecd/stg_oecd__ip_charges.csv` |

**Transformations applied:**
- Parse SDMX-JSON observation keys into flat columns
- Resolve counterpart country ISO codes to full names (from OECD codelist embedded in the response)
- Drop non-actual observations (`OBS_STATUS != 'A'`)
- Validate that constant dimensions (REF_AREA, MEASURE, etc.) are always as expected
- Rename and select final columns; cast types

## 1. Setup

In [1]:
import json
import pandas as pd
from pathlib import Path
import sys

project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root))

from utils.paths import RAW_OECD_DIR, STG_OECD_DIR

STG_OECD_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
INPUT_FILENAME  = 'raw_oecd__ip_charges.json'
OUTPUT_FILENAME = 'stg_oecd__ip_charges.csv'

# Constant dimensions validated BEFORE filtering. MEASURE is intentionally
# excluded here: the OECD API returns parent aggregates (S, CA) alongside the
# filtered SH measure, so MEASURE is not constant in the raw response.
# clean_ip_charges() handles MEASURE filtering explicitly.
EXPECTED = {
    'REF_AREA':         'CAN',
    'ACCOUNTING_ENTRY': 'B',
    'FREQ':             'A',
    'UNIT_MEASURE':     'XDC',   # XDC = national/domestic currency (CAD for Canada)
}

## 2. Import utilities

Parsing and cleaning logic lives in `utils/oecd_utils.py` so it can be
unit-tested independently of the notebook.  See that module for full
implementation details.

In [ ]:
from utils.oecd_utils import parse_sdmx_json, clean_ip_charges

In [4]:
input_path = RAW_OECD_DIR / INPUT_FILENAME
with open(input_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

df = parse_sdmx_json(raw_data)
print(f'Parsed {len(df)} observations')
print(f'Columns: {list(df.columns)}')
df.head()

Parsed 1323 observations
Columns: ['REF_AREA', 'COUNTERPART_AREA', 'counterpart_country', 'MEASURE', 'ACCOUNTING_ENTRY', 'FS_ENTRY', 'FREQ', 'UNIT_MEASURE', 'ADJUSTMENT', 'TIME_PERIOD', 'value_cad_millions', 'obs_status']


,REF_AREA,COUNTERPART_AREA,counterpart_country,MEASURE,ACCOUNTING_ENTRY,FS_ENTRY,FREQ,UNIT_MEASURE,ADJUSTMENT,TIME_PERIOD,value_cad_millions,obs_status
0,CAN,MEX,Mexico,SH,B,T,A,XDC,N,2023,329.0,A
1,CAN,NLD,Netherlands,SH,B,T,A,XDC,N,2023,-329.0,A
2,CAN,NZL,New Zealand,SH,B,T,A,XDC,N,2023,14.0,A
3,CAN,NOR,Norway,SH,B,T,A,XDC,N,2023,-7.0,A
4,CAN,POL,Poland,SH,B,T,A,XDC,N,2023,10.0,A


## 3. Validate constant dimensions

These dimensions are fixed by the API query. Asserting their values here ensures
the API didn't return unexpected data (e.g. a different measure or currency).

In [5]:
for col, expected_val in EXPECTED.items():
    unique_vals = df[col].unique()
    assert len(unique_vals) == 1 and unique_vals[0] == expected_val, (
        f'Unexpected values in {col}: {unique_vals} (expected only "{expected_val}")'
    )
print('Dimension validation passed.')

Dimension validation passed.


## 4. Clean

In [ ]:
df = clean_ip_charges(df)

print(f'Final shape: {df.shape}')
print(f'Years:       {sorted(df["year"].dropna().unique().tolist())}')
df.head(10)

## 5. Save

In [7]:
output_path = STG_OECD_DIR / OUTPUT_FILENAME
df.to_csv(output_path, index=False, encoding='utf-8')
print(f'Staged data saved to: {output_path}')
print(f'Shape: {df.shape}')

Staged data saved to: C:\Users\GuidaK\OneDrive - ISED-ISDE\Documents\Work\or_country_profiles_dashboard\data\staging\oecd\stg_oecd__ip_charges.csv
Shape: (1315, 4)
